## CLI entry points

In [ ]:
#| default_exp cli

### Command wrappers

The implementation notebooks keep the notebook logic. This module owns the installed command-line surface: `@call_parse` entry points, script-shaped argument annotations, and command tracking.

In [ ]:
#| export
import json, os, shlex, signal, subprocess, time, traceback

from contextlib import contextmanager
from functools import wraps
from pathlib import Path
from typing import Annotated

from fastcore.script import call_parse

import nbskill.convert
import nbskill.execute
import nbskill.graph
import nbskill.knowledge
import nbskill.mcp
import nbskill.read
import nbskill.review
import nbskill.nbskill
import nbskill.workbench
import nbskill.write
from nbskill.foundation import failure_map_path, install_nbdev_pre_commit_hooks, load_failure_map

In [ ]:
#| export
_NBSKILL_HOOK_ROOTS = set()


In [ ]:
#| export
def _bump_count(data, kind, tool):
    counts = data.setdefault("counts", {})
    group = counts.setdefault(kind, {})
    group[tool] = group.get(tool, 0) + 1


In [ ]:
#| export
def _call_details(args, kwargs):
    details = {"cwd": str(Path.cwd())}
    if args and isinstance(args[0], (str, Path)): details["path"] = str(args[0])
    for key in ("path", "cell_id", "chapter"):
        value = kwargs.get(key)
        if value is not None: details[key] = str(value)
    return details


In [ ]:
#| export
def _write_failure_map(path, data):
    data["events"] = data.get("events", [])[-200:]
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")


In [ ]:
#| export
def _record_tool_start(tool, details=None):
    path = failure_map_path()
    now = time.time()
    details = details or {}
    event = {"tool": tool, "ts": now, **details}
    try:
        data = load_failure_map(path)
        _bump_count(data, "usage", tool)
        last = data.get("last_call")
        if last:
            delta = now - float(last.get("ts", now))
            reasons = []
            if last.get("tool") == tool: reasons.append("same_tool")
            if delta <= 1.0: reasons.append("within_1s")
            if reasons:
                _bump_count(data, "friction", tool)
                data["events"].append({
                    "kind": "friction",
                    "tool": tool,
                    "path": details.get("path"),
                    "cell_id": details.get("cell_id"),
                    "previous_tool": last.get("tool"),
                    "previous_path": last.get("path"),
                    "seconds_since_previous": round(delta, 3),
                    "reasons": reasons,
                    "ts": now,
                })
        data["last_call"] = event
        _write_failure_map(path, data)
    except OSError:
        pass
    return event


In [ ]:
#| export
def _record_tool_failure(event, exc):
    path = failure_map_path()
    try:
        data = load_failure_map(path)
        tool = event["tool"]
        summary = "".join(traceback.format_exception_only(type(exc), exc)).strip()
        _bump_count(data, "failures", tool)
        data["events"].append({
            "kind": "failure",
            "tool": tool,
            "path": event.get("path"),
            "cell_id": event.get("cell_id"),
            "chapter": event.get("chapter"),
            "cwd": event.get("cwd"),
            "error_type": type(exc).__name__,
            "error": str(exc),
            "summary": summary,
            "ts": time.time(),
        })
        _write_failure_map(path, data)
    except OSError:
        pass


In [ ]:
#| export
@contextmanager
def _track_tool(tool, details=None):
    event = _record_tool_start(tool, details=details)
    try:
        yield
    except BaseException as exc:
        _record_tool_failure(event, exc)
        raise


In [ ]:
#| export
def _ensure_nbdev_pre_commit_hooks(path="."):
    if os.environ.get("NBSKILL_NO_INSTALL_HOOKS"): return None
    root = Path.cwd()
    if str(root) in _NBSKILL_HOOK_ROOTS: return None
    _NBSKILL_HOOK_ROOTS.add(str(root))
    try: return install_nbdev_pre_commit_hooks(path)
    except BaseException: return None


In [ ]:
#| export
def tracked_call(func):
    "Record one command-line tool call and ensure nbdev hooks are installed."
    @wraps(func)
    def wrapper(*args, **kwargs):
        _ensure_nbdev_pre_commit_hooks()
        with _track_tool(func.__name__, details=_call_details(args, kwargs)):
            return func(*args, **kwargs)
    return wrapper


In [ ]:
#| export
def _print_json(value):
    print(json.dumps(value, indent=2, sort_keys=True))
    return None


In [ ]:
#| export
def _print_workbench_result(result):
    "Print a rendered workbench plan or compact execution score."
    if "score" in result:
        score = result.get("score", {})
        payload = {
            "summary": result.get("summary"),
            "passed": score.get("passed"),
            "hard_failures": score.get("hard_failures", []),
            "empirical_warnings": score.get("empirical_warnings", []),
        }
        print(json.dumps(payload, indent=2, sort_keys=True))
        return None
    text = (
        result.get("rendered_plan")
        or result.get("summary")
        or json.dumps(result, indent=2, sort_keys=True)
    )
    print(text)
    return None

In [ ]:
#| hide
from contextlib import redirect_stdout
from io import StringIO

out = StringIO()
with redirect_stdout(out):
    _print_workbench_result({
        "summary": "agent_workbench executed plan",
        "score": {
            "passed": False,
            "hard_failures": [{"code": "empirical_missing_scratch"}],
            "empirical_warnings": [{"code": "empirical_missing_scratch"}],
        },
    })
printed = json.loads(out.getvalue())
assert printed["passed"] is False
assert printed["hard_failures"][0]["code"] == "empirical_missing_scratch"
assert printed["empirical_warnings"][0]["code"] == "empirical_missing_scratch"

In [ ]:
#| export
def _format_status(data):
    lines = [
        "nbskill status",
        "status_transport=local CLI inspection; no active MCP stdio connection required",
        f"version={data['version']}",
        f"cwd={data['cwd']}",
        f"python={data['python']}",
        f"mcp_command={data['mcp_command']}",
        f"mcp_command_path={data['mcp_command_path'] or '(not on PATH)'}",
        "cli_tools:",
    ]
    lines.extend(f"- {name}: {path or '(not on PATH)'}" for name, path in data["cli_tools"].items())
    lines.append(f"reconnect_hint={data['reconnect_hint']}")
    lines.append("restart_hint=nbskill_mcp_restart stops stdio servers; reconnect the MCP client to start a fresh nbskill_mcp")
    lines.append("install_commands:")
    lines.extend(f"- {cmd}" for cmd in data["install_commands"])
    return "\n".join(lines)

In [ ]:
#| export
@call_parse
@tracked_call
def context(
    target: str = "project",  # project, notebook path/name, chapter title, cell id, Python symbol, or literal search text
    scope: str = ".",  # Project, folder, glob, or notebook used to narrow target lookup
    mode: str = "auto",  # auto, overview, edit, or review
):
    "Show project, notebook, chapter, cell, or symbol context in source order."
    nbskill.read.context(target=target, scope=scope, mode=mode)

In [ ]:
#| export
@call_parse
@tracked_call
def filter_context(
    scope: str = ".",  # Project, folder, glob, or notebook used to choose notebooks
    query: str | None = None,  # id/type/chapter/contains/regex/errors/export/headers selectors
    include_re: str | None = None,  # Optional regex that matched cell source must include
    exclude_re: str | None = None,  # Optional regex that matched cell source must not include
    max_matches: int = 50,  # Maximum matching cells to return in source order
    max_chars_per_cell: int = 1200,  # Maximum source characters shown per cell
    view: str = "source",  # source, summary, or cell
    line_numbers: bool = False,  # Include 1-based line numbers in source views
    before: int = 0,  # Number of preceding neighbor cells to summarize
    after: int = 0,  # Number of following neighbor cells to summarize
):
    "Search notebook cells in source order and view their local Docs/example/test neighborhood."
    nbskill.read.filter_context(
        scope=scope, query=query, include_re=include_re, exclude_re=exclude_re,
        max_matches=max_matches, max_chars_per_cell=max_chars_per_cell,
        view=view, line_numbers=line_numbers, before=before, after=after,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def write_nb(
    path: str,  # Notebook path
    cells: Annotated[str, "Cell block text", {"opt": False, "nargs": "?"}] = "",  # Cells to write; use - to read stdin
    cells_file: str | None = None,  # Read cell block text from a UTF-8 file to avoid shell escaping
    before_id: str | None = None,  # Insert before this stable cell id
    after_id: str | None = None,  # Insert after this stable cell id
    chapter: str | None = None,  # Chapter title string or regex; missing chapters are created
    replace: bool = False,  # Replace the full notebook, or the selected chapter body
    cell_type: str = "code",  # Default type for cells without %% marker
    run_test: bool = False,  # Execute the notebook with execnb after writing
    run_style: bool = False,  # Run chstyle after writing
    style_strict: bool = False,  # Fail when chstyle finds hints
    validate_code: bool = True,  # Validate new Python code cells before writing
):
    "Write cells to a notebook."
    return nbskill.write.write_nb(
        path, cells=cells, cells_file=cells_file, before_id=before_id, after_id=after_id, chapter=chapter,
        replace=replace, cell_type=cell_type, run_test=run_test, run_style=run_style, style_strict=style_strict,
        validate_code=validate_code,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def replace_str_nb(
    path: str,  # Notebook path, directory, or glob
    old_str: str,  # Literal text to replace across notebook cell sources
    new_str: str,  # Literal replacement text
    run_test: bool = False,  # Execute the notebook with execnb after writing
    validate_code: bool = True,  # Validate changed Python code cells before writing
    dry_run: bool = False,  # Show replacement plan without writing
    show_cells: bool = False,  # Include touched cell ids and compact diffs
):
    "Replace literal text across notebook cell sources."
    return nbskill.write.write_literal_replacements(
        path, old_str=old_str, new_str=new_str, run_test=run_test, validate_code=validate_code,
        dry_run=dry_run, show_cells=show_cells,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def update_cell(
    path: str,  # Notebook path
    new: Annotated[str, "Replacement cell source, replacement text, or line-range replacement", {"opt": False, "nargs": "?"}] = "",
    new_file: str | None = None,  # Read replacement text from a UTF-8 file
    decode_newlines: bool = True,  # Decode literal \n sequences from CLI text
    cell_id: str | None = None,  # Stable cell id
    old_str: str | None = None,  # Literal text to replace in the selected cell
    line_range: str | None = None,  # 1-based line or range, such as 2 or 2:4
    split: bool = False,  # Split a multi-cell replacement into separate cells
    split_before: str | None = None,  # Split the existing cell before the first matching line
    cell_type: str = "code",  # Replacement cell type when parsing text
    run_test: bool = False,  # Execute the notebook after writing
    validate_code: bool = True,  # Validate replacement Python
    dry_run: bool = False,  # Show the edit without writing
):
    "Update one notebook cell by id, replace text/ranges, or split one cell."
    return nbskill.write.update_cell(
        path, new=new, new_file=new_file, decode_newlines=decode_newlines, cell_id=cell_id, old_str=old_str,
        line_range=line_range, split=split, split_before=split_before, cell_type=cell_type, run_test=run_test,
        validate_code=validate_code, dry_run=dry_run,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def batch_edit_nb(
    plan: Annotated[str, "JSON edit plan, or - to read stdin", {"opt": False, "nargs": "?"}] = "",
    plan_file: str | None = None,  # Read JSON edit plan from a UTF-8 file
    path: str | None = None,  # Default notebook path for operations without a path
    dry_run: bool = True,  # Show the edit plan without writing
    validate_code: bool = True,  # Validate replacement Python
    default_cell_type: str = "code",  # Default type for structured cells
):
    "Apply a JSON batch edit plan to one or more notebooks with locks, diffs, and read-back verification."
    return nbskill.write.batch_edit_nb(
        plan=plan, plan_file=plan_file, path=path, dry_run=dry_run, validate_code=validate_code,
        default_cell_type=default_cell_type,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def split_nb_chapter(
    path: str,  # Source notebook path
    chapter: str,  # Chapter title string or regex to split out
    dest: str,  # Destination notebook path
    default_exp: str | None = None,  # Destination nbdev default_exp; defaults from dest path
    dry_run: bool = True,  # Show the split plan without writing notebooks
    force: bool = False,  # Overwrite dest if it already exists
    promote_private: bool = True,  # Promote referenced private source helpers by dropping the leading underscore
):
    "Split one ## chapter into a new nbdev notebook."
    return nbskill.write.split_nb_chapter(
        path, chapter, dest, default_exp=default_exp, dry_run=dry_run, force=force, promote_private=promote_private,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def exec_nb(
    path: str,  # Notebook path
    up2id: int | str | None = None,  # Stop after this cell index or id
    chapter: str | None = None,  # Run one chapter by heading
    timeout: int = 30,  # Per-cell timeout in seconds
    show_output: bool = True,  # Print visible outputs after execution
    allow_new: bool = False,  # Permit new code cells without prior execution approval
    check_only: bool = False,  # Execute without writing outputs back
    safe: bool = True,  # Use safepyrun safe mode
    allow: str | None = None,  # Comma-separated trusted callables to allow
    ok_dests: str | None = None,  # Comma-separated allowed write destinations
):
    "Execute a notebook with the normal safe defaults."
    return nbskill.execute.exec_nb(
        path, up2id=up2id, chapter=chapter, timeout=timeout, show_output=show_output,
        allow_new=allow_new, check_only=check_only, safe=safe, allow=allow, ok_dests=ok_dests,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def diff_nb(
    path: str,  # Notebook path
    ref_a: str | None = "HEAD",  # Git ref for the left side; pass None to diff the file against itself
    ref_b: str | None = None,  # Git ref for the right side; defaults to working tree
    adds: bool = True,  # Include added cells
    changes: bool = True,  # Include changed cells
    dels: bool = False,  # Include deleted cells
    cell_id: str | None = None,  # Restrict output to one cell id
    after_id: str | None = None,  # Restrict output to cells after this id
    surface: str = "code",  # Review surface: "code" or "public-ui"
):
    "Print notebook diffs for code cells or likely public UI text."
    return nbskill.review.diff_nb(
        path,
        ref_a=ref_a,
        ref_b=ref_b,
        adds=adds,
        changes=changes,
        dels=dels,
        cell_id=cell_id,
        after_id=after_id,
        surface=surface,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def style_check(
    path: Annotated[str, "File or folder to check", {"opt": False, "nargs": "?"}] = ".",  # File or folder to check
    skip_folder_re: str | None = None,  # Regex for folders to skip
    skip_path: str | None = None,  # Comma-separated paths to skip
    strict: bool = False,  # Exit non-zero when diagnostics are present
    delete_after_output: bool = False,  # Compatibility spelling for delete-after-output
    delete_after_outout: bool = False,  # Deprecated misspelling kept for CLI compatibility
    max_output_chars: int = 12000,  # Cap printed output
    max_diagnostics: int = 200,  # Cap structured diagnostics
    fix: bool = False,  # Apply safe automatic fixes
    dry_run: bool = True,  # Show fixes without applying them
    changed_only: bool = False,  # Restrict notebook style diagnostics to changed cells
    ref_a: str | None = "HEAD",  # Left git ref for changed-only mode
    ref_b: str | None = None,  # Right git ref for changed-only mode
):
    "Print capped fast.ai style hints, notebook hygiene warnings, and global tool usage."
    return nbskill.review.style_check(
        path=path, skip_folder_re=skip_folder_re, skip_path=skip_path, strict=strict,
        delete_after_output=delete_after_output, delete_after_outout=delete_after_outout,
        max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, fix=fix, dry_run=dry_run,
        changed_only=changed_only, ref_a=ref_a, ref_b=ref_b,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def nbskill_validate(
    path: Annotated[str, "Notebook file, folder, or glob to validate", {"opt": False, "nargs": "?"}] = "nbs",
    strict: bool = True,  # Exit non-zero when validation problems are present
):
    "Validate nbskill metadata needed for safe notebook tools."
    return nbskill.review.validate_nbs(path=path, strict=strict)

In [ ]:
#| export
@call_parse
@tracked_call
def convert(
    path: str,  # Python file/folder, or existing Python project/package root
    mode: str = "notebook",  # notebook for file/folder conversion, project for nbdev project creation
    dest: str | None = None,  # Notebook path/output folder for notebook mode; project root for project mode
    nbs_path: str = "nbs",  # Destination notebooks folder
    recursive: bool = True,  # Search subfolders in notebook mode
    maxdepth: int | None = None,  # Maximum folder scan depth in notebook mode
    preserve_tree: bool = True,  # Preserve package folder structure under nbs_path
    class_lines: int = 100,  # Split classes larger than this line count
    method_lines: int = 10,  # Split methods larger than this out of large classes
    package: str | None = None,  # Package root name
    include: str | None = None,  # Comma-separated include globs
    exclude: str | None = None,  # Comma-separated exclude globs
    skip_init: bool = True,  # Skip __init__.py files in notebook mode
    include_tests: bool = False,  # Include test files in notebook mode
    dry_run: bool = False,  # Show planned writes without writing
    force: bool = True,  # Overwrite existing notebooks/files
    run_validation: bool = True,  # Run nbdev validation in project mode
):
    "Convert Python sources to nbdev notebooks or a small nbdev project."
    return nbskill.convert.convert(
        path, mode=mode, dest=dest, nbs_path=nbs_path, recursive=recursive, maxdepth=maxdepth,
        preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines,
        package=package, include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests,
        dry_run=dry_run, force=force, run_validation=run_validation,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def build_nbskill_skill(readme_path: str = "README.md", out_path: str = "nbskill/SKILL.md"):
    "Build SKILL.md from the marked section of README.md."
    return nbskill.nbskill.build_skill_from_readme(readme_path=readme_path, out_path=out_path)

In [ ]:
#| export
@call_parse
@tracked_call
def install_nbskill(
    target: str = "codex",  # Target agent: codex, claude, cursor, or both
    skills_dir: str | None = None,  # Explicit skills directory
    skill_name: str = "jupyter-notebooks",  # Installed skill folder name
    overwrite: bool = True,  # Overwrite an existing installation
    install_hooks: bool = False,  # Install nbdev pre-commit hooks
    restart_mcp: bool = True,  # Include restart guidance for running MCP servers
    cursor_workspace: str | None = None,  # Cursor workspace for .cursor/mcp.json; omit for global ~/.cursor/mcp.json
    reference_roots: str = "~/projects",  # Local Git repositories to index
    index_references: bool = True,  # Index references during installation
):
    "Install nbskill agent instructions and configure reusable references."
    return nbskill.nbskill.install_nbskill(
        target=target, skills_dir=skills_dir, skill_name=skill_name, overwrite=overwrite,
        install_hooks=install_hooks, restart_mcp=restart_mcp, cursor_workspace=cursor_workspace,
        reference_roots=reference_roots, index_references=index_references,
    )

In [ ]:
#| export
@call_parse
@tracked_call
def symbol_connection(path: str = "nbs", start: str = "", end: str = "", max_depth: int = 6, json_output: bool = False):
    "Print the shortest static callee chain connecting two notebook symbols."
    return nbskill.graph.symbol_connection(path=path, start=start, end=end, max_depth=max_depth, json_output=json_output)

In [ ]:
#| export
@call_parse
@tracked_call
def reference(
    action: str = "query",  # add, discover, list, ingest, propose, or query
    query: str | None = None,  # Natural-language query for action=query
    top_k: int = 3,  # Number of implementation hits for action=query
    include_branch: bool = False,  # Include direct same-repo callers and callees
    current_repo: str = ".",  # Current project for dependency status
    repos: str | None = None,  # Optional repo name or comma-separated names to search
    repo: str | None = None,  # Reference repository for action=propose
    url: str | None = None,  # Repository URL/path for action=add
    name: str | None = None,  # Reference name for add/ingest
    version: str | None = None,  # Git ref for add; package version for query/propose
    package: str | None = None,  # Package filter or package name
    path: str | None = None,  # Override reference home
    roots: str | None = None,  # Comma-separated local roots for action=discover
    ingest: bool = True,  # Index discovered repositories
    problem: str | None = None,  # Problem statement for action=propose
    kind: str | None = None,  # Optional query filter: readme, module, function, class, method
    module: str | None = None,  # Optional module filter for action=query
    symbol: str | None = None,  # Optional symbol filter for action=query/propose
    include_local: bool = True,  # Include current repository reuse_advice matches
    candidate_k: int | None = None,  # Candidate pool size before final reranking
    explain: bool = True,  # Include score breakdowns and why text
    all: bool = False,  # Ingest all registered references
    force: bool = False,  # Re-ingest even when the indexed version matches
    allow_download: bool = True,  # Download missing requested package versions for action=query
):
    "Manage and search reference implementations."
    action = str(action or "query").lower()
    if action == "add":
        if not url: raise ValueError("reference action='add' needs url")
        result = nbskill.knowledge.reference_add(url, name=name, version=version or "HEAD", package=package, path=path)
    elif action == "discover":
        result = nbskill.knowledge.reference_discover(roots=roots, path=path, ingest=ingest, force=force)
    elif action == "list":
        result = nbskill.knowledge.reference_list(path=path)
    elif action == "ingest":
        result = nbskill.knowledge.reference_ingest(name=name, all=all, path=path, force=force)
    elif action == "propose":
        if not problem or not repo: raise ValueError("reference action='propose' needs problem and repo")
        result = nbskill.knowledge.reference_propose(problem, repo=repo, version=version, symbol=symbol, current_repo=current_repo, path=path)
    elif action == "query":
        if not query: raise ValueError("reference action='query' needs query")
        result = nbskill.knowledge.reference_query(
            query, top_k=top_k, include_branch=include_branch, current_repo=current_repo, repos=repos, path=path,
            kind=kind, package=package, version=version, module=module, symbol=symbol,
            include_local=include_local, candidate_k=candidate_k, explain=explain, allow_download=allow_download,
        )
    else:
        raise ValueError("action must be add, discover, list, ingest, propose, or query")
    return _print_json(result)

In [ ]:
#| hide
import inspect

_exec_nb_target = getattr(exec_nb, "__wrapped__", exec_nb)
_exec_nb_params = inspect.signature(_exec_nb_target).parameters
for _name in ("safe", "allow", "ok_dests"):
    assert _name in _exec_nb_params

_reference_target = getattr(reference, "__wrapped__", reference)
_reference_params = inspect.signature(_reference_target).parameters
for _name in ("include_local", "candidate_k", "explain"):
    assert _name in _reference_params

In [ ]:
#| export
@call_parse
@tracked_call
def problem_memory(
    action: str = "query",  # add, list, or query
    query: str | None = None,  # Natural-language problem query for action=query
    top_k: int = 5,  # Number of memories for action=query
    problem: str | None = None,  # Problem statement for action=add
    solution: str | None = None,  # Solution statement for action=add
    task: str = "",  # Task where the pair was observed
    project: str | None = None,  # Project path/label filter for query, or row project for add
    evidence: str = "",  # Short evidence the solution worked
    outcome: str = "applied",  # applied, partial, failed, skipped, etc.
    tags: str | None = None,  # At least four namespaced tags
    repository: str = "",  # Repository containing the verified evidence
    commit: str = "",  # Source commit for the evidence
    verification: str = "",  # Command that verified the solution
    verified_at: float | None = None,  # Verification time as a Unix timestamp
    path: str | None = None,  # Override reference/problem-memory home
    limit: int = 20,  # Number of recent memories for action=list
):
    "Manage reusable problem-solution memories."
    action = str(action or "query").lower()
    if action == "add":
        if not problem or not solution: raise ValueError("problem_memory action='add' needs problem and solution")
        result = nbskill.knowledge.problem_statement_add(
            problem, solution, task=task, project=project or ".",
            evidence=evidence, outcome=outcome, tags=tags, repository=repository, commit=commit,
            verification=verification, verified_at=verified_at, path=path,
        )
    elif action == "list":
        result = nbskill.knowledge.problem_statement_list(path=path, limit=limit)
    elif action == "query":
        if not query: raise ValueError("problem_memory action='query' needs query")
        result = nbskill.knowledge.problem_statement_query(query, top_k=top_k, project=project, tags=tags, path=path)
    else:
        raise ValueError("action must be add, list, or query")
    return _print_json(result)

In [ ]:
#| hide
_problem_memory_target = getattr(problem_memory, "__wrapped__", problem_memory)
_problem_memory_params = inspect.signature(_problem_memory_target).parameters
for _name in ("problem", "solution", "project", "limit"):
    assert _name in _problem_memory_params

In [ ]:
#| export
@call_parse
@tracked_call
def agent_workbench(
    goal: str,  # Desired software-development outcome
    notebook: str | None = None,  # Required when execute=True; also narrows context when provided
    contract_file: str | None = None,  # Optional JSON contract overrides
    execute: bool = False,  # Execute the rendered plan through execute_plan
    max_steps: int = 8,  # Maximum inner-agent steps when executing
    timeout: int = 30,  # Per-cell timeout for execute_plan
    enforce_empirical: bool = False,  # Promote empirical-loop warnings to hard failures
):
    "Prepare or execute a taste-aware, small-diff agent workbench run."
    result = nbskill.workbench.agent_workbench(
        goal, notebook=notebook, contract_file=contract_file, execute=execute,
        max_steps=max_steps, timeout=timeout,
        enforce_empirical=enforce_empirical,
    )
    return _print_workbench_result(result)


In [ ]:
#| export
@call_parse
@tracked_call
def nbskill_status(json_output: bool = False):  # Print JSON instead of text
    "Report nbskill version, MCP command setup, canonical CLI tools, and reconnect hints."
    data = nbskill.mcp.nbskill_status(json_output=json_output)
    print(json.dumps(data, indent=2, sort_keys=True) if json_output else _format_status(data))
    return None


In [ ]:
#| export
@call_parse
@tracked_call
def nbskill_mcp_log(
    path: str | None = None,  # MCP JSONL log path; defaults to NBSKILL_MCP_LOG or ~/.nbskill-mcp.jsonl
    limit: int = 20,  # Maximum problem and tool rows to print
    all_problems: bool = False,  # Print every problem row instead of applying limit
    until_line: int | None = None,  # Ignore log rows after this 1-based line number
    until_ts: str | None = None,  # Ignore log rows after this ISO timestamp
    json_output: bool = False,  # Print JSON instead of text
):
    "Print concise nbskill MCP log metrics and recent problems."
    report_limit = 0 if all_problems else limit
    report = nbskill.mcp.mcp_log_report(path=path, limit=report_limit, until_line=until_line, until_ts=until_ts)
    text = nbskill.mcp.format_mcp_log_report(report, limit=report_limit)
    print(json.dumps(report, indent=2, sort_keys=True) if json_output else text)
    return None


@call_parse
@tracked_call
def nbskill_mcp_log_problems(
    path: str | None = None,  # MCP JSONL log path; defaults to NBSKILL_MCP_LOG or ~/.nbskill-mcp.jsonl
    limit: int = 20,  # Maximum problem rows to print
    all_problems: bool = False,  # Print every problem row instead of applying limit
    until_line: int | None = None,  # Ignore log rows after this 1-based line number
    until_ts: str | None = None,  # Ignore log rows after this ISO timestamp
    json_output: bool = False,  # Print JSON instead of text
):
    "Print only the problem-focused nbskill MCP log summary."
    report_limit = 0 if all_problems else limit
    report = nbskill.mcp.mcp_log_report(path=path, limit=report_limit, until_line=until_line, until_ts=until_ts)
    text = nbskill.mcp.format_mcp_log_report(report, limit=report_limit, problems_only=True)
    print(json.dumps(report, indent=2, sort_keys=True) if json_output else text)
    return None

In [ ]:
#| export
@call_parse
def nbskill_mcp(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local servers
    show_banner: bool = False,  # Show FastMCP startup banner
    reload: bool = False,  # Restart the FastMCP worker when source files change
    reload_dir: str | None = None,  # Directory to watch for source changes
):
    "Run the nbskill MCP server."
    return nbskill.mcp.main(transport=transport, show_banner=show_banner, reload=reload, reload_dir=reload_dir)

### MCP server process control
Codex owns stdio MCP server lifetimes, but nbskill can still make development safer by finding and stopping the per-project server processes it launched. Stopping the old process lets the MCP client reconnect to a fresh `nbskill_mcp` after exports or package updates, without changing the shared knowledge database.

`find_nbskill_mcp_processes` and `stop_nbskill_mcp_processes` are public because the skill installer needs the same process-control behavior as the CLI.

In [ ]:
#| export
_MCP_CONTROL_COMMANDS = {"nbskill_mcp_start", "nbskill_mcp_stop", "nbskill_mcp_restart"}

In [ ]:
#| export
def _mcp_process_rows():
    proc = subprocess.run(["ps", "-axo", "pid=,ppid=,command="], text=True, capture_output=True)
    if proc.returncode:
        raise RuntimeError((proc.stderr or proc.stdout or "ps failed").strip())
    rows = []
    for line in proc.stdout.splitlines():
        parts = line.strip().split(None, 2)
        if len(parts) != 3: continue
        try: pid, ppid = int(parts[0]), int(parts[1])
        except ValueError: continue
        rows.append({"pid": pid, "ppid": ppid, "command": parts[2]})
    return rows

In [ ]:
#| export
def _process_cwd(pid):
    proc_cwd = Path("/proc") / str(pid) / "cwd"
    try:
        if proc_cwd.exists(): return str(proc_cwd.resolve())
    except OSError:
        pass
    try: proc = subprocess.run(["lsof", "-a", "-p", str(pid), "-d", "cwd", "-Fn"], text=True, capture_output=True)
    except OSError: return None
    if proc.returncode: return None
    for line in proc.stdout.splitlines():
        if line.startswith("n") and line[1:]: return line[1:]
    return None

In [ ]:
#| export
def _command_tokens(command):
    try: return shlex.split(command)
    except ValueError: return command.split()

In [ ]:
#| export
def _mcp_command_is_nbskill_server(command):
    tokens = _command_tokens(command)
    names = {Path(token).name for token in tokens}
    if names & _MCP_CONTROL_COMMANDS: return False
    if "nbskill_mcp" in names: return True
    if "-m" in tokens and "nbskill.mcp" in tokens: return True
    return "fastmcp" in names and "nbskill.mcp" in command

In [ ]:
#| export
def _mcp_project_root(project=None):
    return Path(project or Path.cwd()).expanduser().resolve()

In [ ]:
#| export
def _mcp_project_match_texts(project=None):
    raw = Path(project or Path.cwd()).expanduser()
    texts = {str(raw)}
    try: texts.add(str(raw.resolve()))
    except OSError: pass
    return texts

In [ ]:
#| export
def _path_is_or_below(path, root):
    try:
        path = Path(path).expanduser().resolve()
        root = Path(root).expanduser().resolve()
    except OSError:
        return False
    return path == root or root in path.parents

In [ ]:
#| export
def _mcp_process_with_cwd(row):
    if row.get("cwd") or not row.get("pid"): return row
    cwd = _process_cwd(row["pid"])
    return {**row, "cwd": cwd} if cwd else row

In [ ]:
#| export
def _mcp_process_matches(row, project=None, all_projects=False):
    command = row.get("command", "")
    if row.get("pid") == os.getpid(): return False
    if not _mcp_command_is_nbskill_server(command): return False
    if all_projects: return True
    root = _mcp_project_root(project)
    if row.get("cwd") and _path_is_or_below(row["cwd"], root): return True
    return any(text in command for text in _mcp_project_match_texts(project))

In [ ]:
#| export
def find_nbskill_mcp_processes(project=None, all_projects=False):
    "Return running nbskill MCP server processes that match a project scope."
    rows = []
    for row in _mcp_process_rows():
        if not _mcp_process_matches(row, all_projects=True): continue
        row = _mcp_process_with_cwd(row)
        if _mcp_process_matches(row, project=project, all_projects=all_projects): rows.append(row)
    return rows

In [ ]:
#| export
def _alive_pids(pids):
    current = {row["pid"] for row in _mcp_process_rows()}
    return [pid for pid in pids if pid in current]

In [ ]:
#| export
def _descendant_rows(rows, parent_pids):
    children_by_parent = {}
    for row in rows: children_by_parent.setdefault(row.get("ppid"), []).append(row)
    descendants, seen, frontier = [], set(parent_pids), list(parent_pids)
    while frontier:
        parent = frontier.pop(0)
        for child in children_by_parent.get(parent, []):
            pid = child.get("pid")
            if not pid or pid in seen or pid == os.getpid(): continue
            seen.add(pid)
            descendants.append(child)
            frontier.append(pid)
    return descendants

In [ ]:
#| export
def _mcp_stop_target_rows(matches, all_rows=None):
    all_rows = _mcp_process_rows() if all_rows is None else all_rows
    descendants = _descendant_rows(all_rows, {row["pid"] for row in matches})
    targets, seen = [], set()
    for row in [*reversed(descendants), *matches]:
        pid = row.get("pid")
        if not pid or pid in seen or pid == os.getpid(): continue
        seen.add(pid)
        targets.append(row)
    return targets

In [ ]:
#| export
def stop_nbskill_mcp_processes(project=None, all_projects=False, timeout=5.0, force=True, dry_run=False):
    "Stop running nbskill MCP server processes and return structured results."
    matches = find_nbskill_mcp_processes(project=project, all_projects=all_projects)
    targets = _mcp_stop_target_rows(matches)
    result = {
        "project": None if all_projects else str(_mcp_project_root(project)),
        "all_projects": bool(all_projects),
        "matched": matches,
        "targets": targets,
        "terminated": [],
        "forced": [],
        "alive": [],
        "dry_run": bool(dry_run),
    }
    if dry_run or not targets: return result
    for row in targets:
        try:
            os.kill(row["pid"], signal.SIGTERM)
            result["terminated"].append(row["pid"])
        except ProcessLookupError:
            pass
    deadline = time.time() + float(timeout)
    pending = _alive_pids([row["pid"] for row in targets])
    while pending and time.time() < deadline:
        time.sleep(0.1)
        pending = _alive_pids(pending)
    if pending and force:
        for pid in pending:
            try:
                os.kill(pid, signal.SIGKILL)
                result["forced"].append(pid)
            except ProcessLookupError:
                pass
        pending = _alive_pids(pending)
    result["alive"] = pending
    return result

In [ ]:
#| export
def _print_mcp_control_result(result, json_output=False):
    if json_output:
        print(json.dumps(result, indent=2, sort_keys=True))
        return None
    lines = [
        f"matched={len(result['matched'])}",
        f"targets={len(result.get('targets', result['matched']))}",
        f"terminated={len(result['terminated'])}",
        f"forced={len(result['forced'])}",
        f"alive={len(result['alive'])}",
    ]
    if result.get("project"): lines.insert(0, f"project={result['project']}")
    if result.get("dry_run"): lines.insert(0, "dry_run=true")
    for row in result["matched"]:
        cwd = f" cwd={row['cwd']}" if row.get("cwd") else ""
        lines.append(f"pid={row['pid']} ppid={row['ppid']}{cwd} command={row['command']}")
    target_pids = {row["pid"] for row in result.get("targets", [])} - {row["pid"] for row in result["matched"]}
    if target_pids: lines.append("target_descendants=" + ",".join(str(pid) for pid in sorted(target_pids)))
    if result.get("start"):
        lines.append(f"reconnect={result['start']}")
    print("\n".join(lines))
    return None

In [ ]:
#| export
@call_parse
def nbskill_mcp_start(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local per-project servers
    show_banner: bool = False,  # Show FastMCP startup banner
    reload: bool = False,  # Restart the FastMCP worker when source files change
    reload_dir: str | None = None,  # Directory to watch for source changes
):
    "Run the nbskill MCP server in the foreground."
    return nbskill.mcp.main(transport=transport, show_banner=show_banner, reload=reload, reload_dir=reload_dir)

In [ ]:
#| export
@call_parse
def nbskill_mcp_stop(
    project: str | None = None,  # Project root to target; defaults to the current working directory
    all_projects: bool = False,  # Stop nbskill MCP servers for every project
    timeout: float = 5.0,  # Seconds to wait after SIGTERM before forcing
    force: bool = True,  # Send SIGKILL to remaining matched processes after timeout
    dry_run: bool = False,  # Show matched processes without stopping them
    json_output: bool = False,  # Print JSON instead of text
):
    "Stop running nbskill MCP server processes for this project."
    result = stop_nbskill_mcp_processes(project=project, all_projects=all_projects, timeout=timeout, force=force, dry_run=dry_run)
    return _print_mcp_control_result(result, json_output=json_output)

In [ ]:
#| export
@call_parse
def nbskill_mcp_restart(
    project: str | None = None,  # Optional project root to target when all_projects is false
    all_projects: bool = True,  # Stop every running nbskill MCP server by default
    timeout: float = 5.0,  # Seconds to wait after SIGTERM before forcing
    force: bool = True,  # Send SIGKILL to remaining matched processes after timeout
    dry_run: bool = False,  # Show matched processes without stopping them
    json_output: bool = False,  # Print JSON instead of text
):
    "Stop nbskill MCP stdio servers; clients must reconnect to start fresh code."
    result = stop_nbskill_mcp_processes(project=project, all_projects=all_projects, timeout=timeout, force=force, dry_run=dry_run)
    result["start"] = (
        "nbskill_mcp_restart only stops stdio MCP server processes. "
        "Reconnect or restart the MCP client to launch a fresh nbskill_mcp; "
        "use nbskill_status for local status without an active stdio server."
    )
    return _print_mcp_control_result(result, json_output=json_output)

### MCP process controls

`find_nbskill_mcp_processes` lists running nbskill MCP servers in one project or every project. `stop_nbskill_mcp_processes` reports its targets and stays non-destructive with `dry_run=True`. `nbskill_mcp_start`, `nbskill_mcp_stop`, and `nbskill_mcp_restart` expose the same controls as command-line entry points.

In [ ]:
find_nbskill_mcp_processes(project="nbs/data")
stop_nbskill_mcp_processes(project="nbs/data", dry_run=True)

In [ ]:
#| eval: false
nbskill_mcp_start(reload=True, reload_dir="nbskill")
nbskill_mcp_stop(dry_run=True)
nbskill_mcp_restart(dry_run=True)

In [ ]:
#| hide
control = find_nbskill_mcp_processes.__globals__
rows = [dict(pid=999991, ppid=1, command="uv run nbskill_mcp", cwd="/tmp/demo")]
old_rows, old_stop, old_print, old_main = control["_mcp_process_rows"], control["stop_nbskill_mcp_processes"], control["_print_mcp_control_result"], nbskill.mcp.main
try:
    control["_mcp_process_rows"] = lambda: rows
    assert find_nbskill_mcp_processes(all_projects=True) == rows
    assert stop_nbskill_mcp_processes(all_projects=True, dry_run=True)["matched"] == rows
    control["stop_nbskill_mcp_processes"] = lambda **kwargs: kwargs
    control["_print_mcp_control_result"] = lambda result, json_output=False: result
    nbskill.mcp.main = lambda **kwargs: kwargs
    assert nbskill_mcp_start(reload=True)["reload"]
    assert nbskill_mcp_stop(dry_run=True)["dry_run"]
    assert nbskill_mcp_restart(dry_run=True)["all_projects"]
finally:
    control["_mcp_process_rows"], control["stop_nbskill_mcp_processes"], control["_print_mcp_control_result"], nbskill.mcp.main = old_rows, old_stop, old_print, old_main

In [ ]:
#| hide
sample = {"pid": 123, "ppid": 1, "command": "/opt/homebrew/bin/uv run --project /tmp/demo nbskill_mcp"}
assert _mcp_process_matches(sample, project="/tmp/demo")
assert not _mcp_process_matches(sample, project="/tmp/other")
assert _mcp_process_matches({"pid": 125, "ppid": 1, "command": "/tmp/venv/bin/nbskill_mcp", "cwd": "/tmp/demo"}, project="/tmp/demo")
assert _mcp_process_matches({"pid": 126, "ppid": 1, "command": "python -m nbskill.mcp", "cwd": "/tmp/demo/subdir"}, project="/tmp/demo")
assert not _mcp_process_matches({"pid": 124, "ppid": 1, "command": "nbskill_mcp_restart"}, all_projects=True)
rows = [
    {"pid": 10, "ppid": 1, "command": "uv run nbskill_mcp"},
    {"pid": 11, "ppid": 10, "command": "python worker"},
    {"pid": 12, "ppid": 11, "command": "python grandchild"},
    {"pid": 20, "ppid": 1, "command": "unrelated"},
]
assert [row["pid"] for row in _mcp_stop_target_rows([rows[0]], rows)] == [12, 11, 10]
_fake_globals = []
for _func in (find_nbskill_mcp_processes, stop_nbskill_mcp_processes, _mcp_stop_target_rows):
    _already_seen = False
    for _globals in _fake_globals:
        if id(_globals) == id(_func.__globals__): _already_seen = True
    if not _already_seen: _fake_globals.append(_func.__globals__)
_originals = [(_globals, _globals["_mcp_process_rows"], _globals.get("_process_cwd")) for _globals in _fake_globals]
try:
    process_rows = [
        {"pid": 999991, "ppid": 1, "command": "uv run nbskill_mcp"},
        {"pid": 31, "ppid": 1, "command": "other"},
    ]
    def _fake_mcp_process_rows(rows=process_rows): return list(rows)
    def _fake_process_cwd(pid, cwd_by_pid={999991: "/tmp/demo"}): return cwd_by_pid.get(pid)
    for _globals in _fake_globals:
        _globals["_mcp_process_rows"] = _fake_mcp_process_rows
        if "_process_cwd" in _globals: _globals["_process_cwd"] = _fake_process_cwd
    found = find_nbskill_mcp_processes(all_projects=True)
    assert [row["pid"] for row in found] == [999991]
    dry = stop_nbskill_mcp_processes(all_projects=True, dry_run=True)
    assert [row["pid"] for row in dry["matched"]] == [999991]
finally:
    for _globals, _mcp_process_rows, _process_cwd in _originals:
        _globals["_mcp_process_rows"] = _mcp_process_rows
        if _process_cwd is not None: _globals["_process_cwd"] = _process_cwd

Fastcore 2 removed `fastcore.script.Param`; CLI wrappers keep positional text arguments by using `typing.Annotated` metadata consumed by `call_parse`.